# Lesson 3.5 — Handling Missing Data

**Objectives**
- Detect missing values with `.isna()` / `.isnull()`
- Decide between dropping (`.dropna()`) and filling (`.fillna()`) missing data
- Understand how missing data will later affect ML models

See `modules/03-pandas/notes.md` (Lesson 3.5) for the full written explanation.


In [1]:
import pandas as pd
from data_science_course.datasets import load_customers, load_orders

customers = load_customers()
orders = load_orders().drop_duplicates(subset=["order_id"])

## Detecting missing values

In [2]:
customers.isna().sum()

customer_id             0
signup_date             0
region                 24
age                    48
membership_tier         0
acquisition_channel     0
churned                 0
dtype: int64

Matches `data/README.md`: `region` ~3% missing (24/800), `age` ~6% missing (48/800).

In [3]:
customers.isna().any(axis=1).sum()   # rows missing something, ANYWHERE

np.int64(72)

In [4]:
customers[customers["region"].isna() & customers["age"].isna()].shape[0]

0

72 rows have at least one gap, but only a smaller number are missing *both* `region`
and `age` -- most rows are missing just one field.

## `.dropna()`: remove rows

In [5]:
print(customers.dropna().shape)                     # drop any row missing ANY column
print(customers.dropna(subset=["region"]).shape)     # only drop rows missing region

(728, 7)
(776, 7)


Dropping any row with any missing value removes 9% of the data -- usually too
aggressive when most of those rows are fine except for one column. `subset=` is
almost always the better default.

## `.fillna()`: keep every row, fill a sensible value

In [6]:
median_age = customers["age"].median()
print(median_age)
customers["age"] = customers["age"].fillna(median_age)
customers["age"].isna().sum()

38.0


np.int64(0)

In [7]:
customers["region"] = customers["region"].fillna("Unknown")
customers["region"].value_counts(dropna=False)

region
East       101
east        86
South       78
south       75
west        72
WEST        70
West        58
north       53
SOUTH       51
 North      51
NORTH       41
North       40
Unknown     24
Name: count, dtype: int64

Numeric columns -> median (resistant to outliers). Categorical columns -> a
placeholder category, so "we don't know" stays visible instead of being merged into
an existing real category.

## A column where "missing" doesn't mean "zero": `discount_pct`

In [8]:
orders.isna().sum()

order_id             0
customer_id          0
product_id           0
order_date           0
quantity             0
unit_price           0
discount_pct       480
payment_method     180
shipping_region      0
status               0
order_total          0
dtype: int64

In [9]:
orders["discount_missing"] = orders["discount_pct"].isna()
orders["discount_pct"] = orders["discount_pct"].fillna(0)
orders[["discount_pct", "discount_missing"]].isna().sum()

discount_pct        0
discount_missing    0
dtype: int64

We filled with 0 *and* kept a flag column -- so later analysis (or a Module 4 model)
can still distinguish "recorded discount of 0%" from "discount not recorded." A silent
`fillna(0)` alone would have erased that distinction.

## Why this matters for ML later

Every scikit-learn model in Module 4 raises a `ValueError` on `.fit()` if any feature
column still has `NaN` values (a few tree-based implementations are exceptions, but
plan around the default). Every drop/fill decision made here becomes part of the
model's behavior later -- not just cosmetic cleanup now. Confirm `customers` and
`orders` are now fully clean of `NaN` in the columns we've fixed:

In [10]:
print(customers[["region", "age"]].isna().sum())
print(orders[["discount_pct"]].isna().sum())

region    0
age       0
dtype: int64
discount_pct    0
dtype: int64


## Try it yourself

1. Check `orders["payment_method"]` for missing values, then fill them with the
   placeholder `"unknown"`.
2. Compare `customers["age"].mean()` computed **before** and **after** filling the
   missing values with the median -- did the mean change much? Why or why not?
3. Instead of filling `age` with the median, try filling it with the **mean**, in a
   fresh copy of `customers` loaded again from `load_customers()`. Which fill value do
   you think is more appropriate here, and why?
4. Drop rows from a **fresh** `load_customers()` copy where `age` is missing, using
   `subset=`, and confirm the resulting shape matches `800 - 48`.


In [11]:
# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO


### Solution

In [12]:
# 1.
print(orders["payment_method"].isna().sum())
orders["payment_method"] = orders["payment_method"].fillna("unknown")
print(orders["payment_method"].isna().sum())

# 2.
fresh = load_customers()
mean_before = fresh["age"].mean()
fresh["age"] = fresh["age"].fillna(fresh["age"].median())
mean_after = fresh["age"].mean()
print(round(mean_before, 3), round(mean_after, 3))
# The mean barely moves -- filling ~6% of rows with a value close to the existing
# mean/median doesn't shift the overall average much, though it does slightly reduce
# the column's variance.

# 3.
fresh2 = load_customers()
fresh2["age"] = fresh2["age"].fillna(fresh2["age"].mean())
print(fresh2["age"].isna().sum())
# Median is usually preferred for a roughly-normal, outlier-light column like age --
# mean and median are close here, so either is defensible, but median is the safer
# general habit since it doesn't shift under skew or outliers.

# 4.
fresh3 = load_customers()
dropped = fresh3.dropna(subset=["age"])
print(dropped.shape, 800 - 48)

180
0
37.471 37.502
0
(752, 7) 752
